# 00a — Data Collection: Real Nepali Financial PDFs

Downloads publicly available Nepali financial reports from official sources.
Converts each PDF page to an image for orientation classification experiments.

## Sources

| Source | Type | URL |
|---|---|---|
| Nepal Rastra Bank | Annual reports, bank supervision | https://www.nrb.org.np |
| SEBON | Capital market annual reports | https://www.sebon.gov.np/annual-report |
| Nepal Bank Ltd | Bank annual reports | https://www.nepalbank.com.np/investor-relations/agm-reports/annual-reports |
| Everest Bank | Annual reports | https://everestbankltd.com/financialreports/annual-report/ |
| MeroLagani | All NEPSE-listed company reports | https://merolagani.com/CompanyReports.aspx?type=ANNUAL |
| Ministry of Finance | Budget speeches | https://mof.gov.np/category/budget-speech/ |

In [ ]:
# pip install pymupdf requests tqdm
import sys
sys.path.insert(0, '..')

import hashlib
import time
from pathlib import Path

import fitz  # PyMuPDF
import requests
from tqdm import tqdm

PDF_DIR  = Path('../data/pdfs')
RAW_DIR  = Path('../data/raw')
PDF_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

DPI = 150  # 150 DPI is sufficient for orientation classification

In [ ]:
# Known direct PDF links from official Nepali sources
# Add more as you find them from the sources listed above
KNOWN_PDFS = [
    {
        'url': 'https://www.nrb.org.np/contents/uploads/2025/03/Annual-Bank-Supervision-Report-2024-4.pdf',
        'name': 'nrb_bank_supervision_2024.pdf',
        'source': 'NRB',
    },
    {
        'url': 'https://demo.mof.gov.np/uploads/document/file/1724825318_Budget%20Speech%202024_25%20English%20FV.pdf',
        'name': 'mof_budget_speech_2024_25.pdf',
        'source': 'Ministry of Finance',
    },
    # Add more URLs here:
    # {'url': '...', 'name': '...', 'source': '...'},
]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; research-bot/1.0)'
}

In [ ]:
def download_pdf(url: str, dest: Path, retries: int = 3) -> bool:
    if dest.exists():
        print(f'  Already downloaded: {dest.name}')
        return True
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
            resp.raise_for_status()
            with open(dest, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f'  Downloaded: {dest.name} ({dest.stat().st_size // 1024} KB)')
            return True
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(2)
    return False

for entry in KNOWN_PDFS:
    print(f"Downloading {entry['name']} from {entry['source']}")
    download_pdf(entry['url'], PDF_DIR / entry['name'])

In [ ]:
# --- Manual downloads ---
# For PDFs that require browser interaction (login, CAPTCHA, JS rendering),
# download them manually and place in data/pdfs/
#
# Recommended manual downloads:
#   - Nabil Bank annual report: https://www.nabilbank.com/investor-relations
#   - NIC Asia annual report: https://www.nicasiabank.com/investors
#   - Everest Bank: https://everestbankltd.com/financialreports/annual-report/
#   - SEBON annual report: https://www.sebon.gov.np/annual-report

pdfs = list(PDF_DIR.glob('*.pdf'))
print(f'Total PDFs in data/pdfs/: {len(pdfs)}')
for p in pdfs:
    print(f'  {p.name}  ({p.stat().st_size // 1024} KB)')

In [ ]:
def pdf_to_images(pdf_path: Path, out_dir: Path, dpi: int = DPI, max_pages: int = 50) -> list[Path]:
    """
    Render each page of a PDF to a PNG image.
    max_pages: cap per PDF to avoid huge datasets from 200-page reports.
    Returns list of output image paths.
    """
    doc = fitz.open(str(pdf_path))
    stem = pdf_path.stem
    paths = []
    pages = min(len(doc), max_pages)
    mat = fitz.Matrix(dpi / 72, dpi / 72)

    for i in range(pages):
        page = doc[i]
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        # Use content hash to avoid duplicates across re-runs
        img_name = f'{stem}_page{i+1:04d}.png'
        img_path = out_dir / img_name
        pix.save(str(img_path))
        paths.append(img_path)

    doc.close()
    return paths


all_image_paths = []

for pdf_path in tqdm(sorted(PDF_DIR.glob('*.pdf')), desc='Converting PDFs'):
    imgs = pdf_to_images(pdf_path, RAW_DIR, max_pages=30)
    all_image_paths.extend(imgs)
    print(f'  {pdf_path.name}: {len(imgs)} pages extracted')

print(f'\nTotal images in data/raw/: {len(all_image_paths)}')

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

# Sanity check: show 6 random pages
samples = random.sample(all_image_paths, min(6, len(all_image_paths)))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(path.name[:40], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample pages from collected PDFs')
plt.tight_layout()
plt.show()